In [2]:
import pandas as pd
import numpy as np

from datetime import timedelta
from pandas.tseries.offsets import MonthEnd

pd.set_option("display.max_columns", None)

df = pd.read_csv(
    "delivered_order_totals_helper.csv",
    sep=";"
)


In [3]:
df.head()

,order_id,customer_unique_id,purchase_date,purchase_month,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,2017-09-13,2017-09-01,72.19
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,2017-04-26,2017-04-01,259.83
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,2018-01-14,2018-01-01,216.87
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,2018-08-08,2018-08-01,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,2017-02-04,2017-02-01,218.04


In [4]:
df.shape

(96478, 5)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            96478 non-null  str    
 1   customer_unique_id  96478 non-null  str    
 2   purchase_date       96478 non-null  str    
 3   purchase_month      96478 non-null  str    
 4   order_value         96478 non-null  float64
dtypes: float64(1), str(4)
memory usage: 3.7 MB


Convert Dates

In [6]:
df["purchase_date"] = pd.to_datetime(df["purchase_date"])

df["purchase_month"] = pd.to_datetime(df["purchase_month"])

Missing Values

In [7]:
df.isnull().sum()

order_id              0
customer_unique_id    0
purchase_date         0
purchase_month        0
order_value           0
dtype: int64

Duplicate Orders

In [8]:
df["order_id"].duplicated().sum()

np.int64(0)

In [9]:
customer_first_purchase = (
    df.groupby("customer_unique_id", as_index=False)
      .agg(first_purchase_date=("purchase_date", "min"))
)

customer_first_purchase["first_purchase_month"] = (
    customer_first_purchase["first_purchase_date"]
      .dt.to_period("M")
      .dt.to_timestamp()
)

customer_first_purchase.head()

,customer_unique_id,first_purchase_date,first_purchase_month
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,2018-05-01
2,0000f46a3911fa3c0805444483337064,2017-03-10,2017-03-01
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12,2017-10-01
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14,2017-11-01


In [10]:
customer_first_purchase.shape

(93358, 3)

Create the Snapshot Months

In [11]:
analysis_end = df["purchase_month"].max()

snapshot_months = pd.date_range(
    start=df["purchase_month"].min(),
    end=analysis_end,
    freq="MS"
)

snapshot_df = pd.DataFrame({
    "snapshot_month_start": snapshot_months
})

snapshot_df

,snapshot_month_start
0,2016-09-01
1,2016-10-01
2,2016-11-01
3,2016-12-01
4,2017-01-01
5,2017-02-01
6,2017-03-01
7,2017-04-01
8,2017-05-01
9,2017-06-01


In [12]:
print(df["purchase_month"].min())
print(df["purchase_month"].max())

2016-09-01 00:00:00
2018-08-01 00:00:00


In [13]:
len(snapshot_months)

24

Create the Customer Snapshot Table

In [14]:
# Create a temporary key for a cross join
customer_first_purchase["key"] = 1
snapshot_df["key"] = 1

# Cross join customers with all snapshot months
customer_snapshots = (
    customer_first_purchase
    .merge(snapshot_df, on="key")
    .drop(columns="key")
)

# Keep only months from the customer's first purchase onwards
customer_snapshots = customer_snapshots[
    customer_snapshots["snapshot_month_start"] >= customer_snapshots["first_purchase_month"]
].copy()

# Add the month end
customer_snapshots["snapshot_month_end"] = (
    customer_snapshots["snapshot_month_start"] + MonthEnd(0)
)

In [15]:
customer_snapshots.head()

,customer_unique_id,first_purchase_date,first_purchase_month,snapshot_month_start,snapshot_month_end
20,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-05-01,2018-05-31
21,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-06-01,2018-06-30
22,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-07-01,2018-07-31
23,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-08-01,2018-08-31
44,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,2018-05-01,2018-05-01,2018-05-31


In [16]:
customer_snapshots.shape

(789093, 5)

Attach Orders to Each Snapshot

In [17]:
customer_orders = customer_snapshots.merge(
    df,
    on="customer_unique_id",
    how="left"
)

customer_orders = customer_orders[
    customer_orders["purchase_date"] <= customer_orders["snapshot_month_end"]
].copy()

customer_orders.shape

(813976, 9)

Calculate All Metrics

In [18]:
customer_metrics = (
    customer_orders
    .groupby(
        [
            "snapshot_month_start",
            "snapshot_month_end",
            "customer_unique_id"
        ],
        as_index=False
    )
    .agg(
        monthly_orders=(
            "order_id",
            lambda x: x[
                customer_orders.loc[x.index, "purchase_month"]
                == customer_orders.loc[x.index, "snapshot_month_start"]
            ].nunique()
        ),

        monthly_monetary_value=(
            "order_value",
            lambda x: round(
                x[
                    customer_orders.loc[x.index, "purchase_month"]
                    == customer_orders.loc[x.index, "snapshot_month_start"]
                ].sum(),
                2
            )
        ),

        cumulative_frequency=("order_id", "nunique"),

        cumulative_monetary_value=("order_value", "sum"),

        last_purchase_date=("purchase_date", "max")
    )
)

customer_metrics["cumulative_monetary_value"] = (
    customer_metrics["cumulative_monetary_value"].round(2)
)

customer_metrics["recency_days"] = (
    customer_metrics["snapshot_month_end"]
    - customer_metrics["last_purchase_date"]
).dt.days

customer_metrics.head()

KeyboardInterrupt: 

Monthly Metrics

In [19]:
monthly_metrics = (
    customer_orders[
        customer_orders["purchase_month"] == customer_orders["snapshot_month_start"]
    ]
    .groupby(
        ["snapshot_month_start", "snapshot_month_end", "customer_unique_id"],
        as_index=False
    )
    .agg(
        monthly_orders=("order_id", "nunique"),
        monthly_monetary_value=("order_value", "sum")
    )
)

monthly_metrics["monthly_monetary_value"] = (
    monthly_metrics["monthly_monetary_value"].round(2)
)

monthly_metrics.shape

(95194, 5)

Calculate the Cumulative Metrics

In [20]:
cumulative_metrics = (
    customer_orders
    .groupby(
        ["snapshot_month_start", "snapshot_month_end", "customer_unique_id"],
        as_index=False
    )
    .agg(
        cumulative_frequency=("order_id", "nunique"),
        cumulative_monetary_value=("order_value", "sum"),
        last_purchase_date=("purchase_date", "max")
    )
)

cumulative_metrics["cumulative_monetary_value"] = (
    cumulative_metrics["cumulative_monetary_value"].round(2)
)

cumulative_metrics["recency_days"] = (
    cumulative_metrics["snapshot_month_end"]
    - cumulative_metrics["last_purchase_date"]
).dt.days

cumulative_metrics.shape

(789093, 7)

Merge the Metrics

In [21]:
customer_segmentation_monthly = (
    customer_snapshots
    .merge(
        monthly_metrics,
        on=[
            "snapshot_month_start",
            "snapshot_month_end",
            "customer_unique_id"
        ],
        how="left"
    )
    .merge(
        cumulative_metrics,
        on=[
            "snapshot_month_start",
            "snapshot_month_end",
            "customer_unique_id"
        ],
        how="left"
    )
)

Replace missing monthly values

Customers who didn't purchase during a month should have zeros.

In [22]:
customer_segmentation_monthly["monthly_orders"] = (
    customer_segmentation_monthly["monthly_orders"]
    .fillna(0)
    .astype(int)
)

customer_segmentation_monthly["monthly_monetary_value"] = (
    customer_segmentation_monthly["monthly_monetary_value"]
    .fillna(0)
)

In [23]:
customer_segmentation_monthly.shape

(789093, 11)

In [24]:
customer_segmentation_monthly.head()

,customer_unique_id,first_purchase_date,first_purchase_month,snapshot_month_start,snapshot_month_end,monthly_orders,monthly_monetary_value,cumulative_frequency,cumulative_monetary_value,last_purchase_date,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-05-01,2018-05-31,1,141.90,1,141.90,2018-05-10,21
1,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-06-01,2018-06-30,0,0.00,1,141.90,2018-05-10,51
2,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-07-01,2018-07-31,0,0.00,1,141.90,2018-05-10,82
3,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-08-01,2018-08-31,0,0.00,1,141.90,2018-05-10,113
4,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,2018-05-01,2018-05-01,2018-05-31,1,27.19,1,27.19,2018-05-07,24


In [25]:
customer_segmentation_monthly.isna().sum()

customer_unique_id           0
first_purchase_date          0
first_purchase_month         0
snapshot_month_start         0
snapshot_month_end           0
monthly_orders               0
monthly_monetary_value       0
cumulative_frequency         0
cumulative_monetary_value    0
last_purchase_date           0
recency_days                 0
dtype: int64

Calculate RFM Scores

In [26]:
customer_segmentation_monthly["recency_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["recency_days"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            5,
            labels=[5,4,3,2,1]
        )
    )
    .astype(int)
)

customer_segmentation_monthly["frequency_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["cumulative_frequency"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            5,
            labels=[1,2,3,4,5]
        )
    )
    .astype(int)
)

customer_segmentation_monthly["monetary_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["cumulative_monetary_value"]
    .transform(
        lambda x: pd.qcut(
            x.rank(method="first"),
            5,
            labels=[1,2,3,4,5]
        )
    )
    .astype(int)
)

ValueError: Bin edges must be unique: Index([1.0, 1.0, 1.0, 1.0, 1.0, 1.0], dtype='float64', name=2016-09-01 00:00:00).
You can drop duplicate edges by setting the 'duplicates' kwarg

Create a SQL-style NTILE function

Calculate the scores

In [34]:
import numpy as np
import pandas as pd

def sql_ntile(series, n=5, higher_is_better=True):
    """
    Replicates SQL NTILE(n) while allowing control over whether
    higher or lower values should receive higher scores.
    """

    # Sort in the correct direction
    order = series.sort_values(
        ascending=not higher_is_better,
        kind="mergesort"
    ).index

    total = len(order)

    # Create NTILE buckets (1 to n)
    buckets = np.floor(np.arange(total) * n / total).astype(int) + 1

    # Reverse the bucket numbers so the "best" values get score 5
    scores = n + 1 - buckets

    return pd.Series(scores, index=order).sort_index()

In [35]:
customer_segmentation_monthly["recency_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["recency_days"]
    .transform(lambda x: sql_ntile(x, higher_is_better=False))
)

customer_segmentation_monthly["frequency_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["cumulative_frequency"]
    .transform(lambda x: sql_ntile(x, higher_is_better=True))
)

customer_segmentation_monthly["monetary_score"] = (
    customer_segmentation_monthly
    .groupby("snapshot_month_start")["cumulative_monetary_value"]
    .transform(lambda x: sql_ntile(x, higher_is_better=True))
)

In [32]:
customer_segmentation_monthly[
    [
        "recency_score",
        "frequency_score",
        "monetary_score"
    ]
].describe()

,recency_score,frequency_score,monetary_score
count,789093.000000,789093.000000,789093.000000
mean,3.000051,3.000051,3.000051
std,1.414213,1.414213,1.414213
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000
50%,3.000000,3.000000,3.000000
75%,4.000000,4.000000,4.000000
max,5.000000,5.000000,5.000000


In [36]:
customer_segmentation_monthly[
    [
        "recency_days",
        "recency_score"
    ]
].sort_values("recency_days").head(20)

,recency_days,recency_score
357775,0,5
786913,0,5
622886,0,5
716724,0,5
614792,0,5
186938,0,5
418058,0,5
339227,0,5
201428,0,5
485383,0,5


Create the RFM Score

In [37]:
customer_segmentation_monthly["rfm_score"] = (
    customer_segmentation_monthly["recency_score"].astype(str)
    + customer_segmentation_monthly["frequency_score"].astype(str)
    + customer_segmentation_monthly["monetary_score"].astype(str)
)

customer segments

In [40]:
def assign_segment(r, f):

    # Champions
    if r >= 4 and f >= 4:
        return "Champions"

    # Loyal Customers
    elif r in [3, 4] and f in [4, 5]:
        return "Loyal Customers"

    # Potential Loyalists
    elif r >= 4 and f in [2, 3]:
        return "Potential Loyalists"

    # New Customers
    elif r == 5 and f == 1:
        return "New Customers"

    # Promising
    elif r == 4 and f == 1:
        return "Promising"

    # Need Attention
    elif r == 3 and f in [2, 3]:
        return "Need Attention"

    # About to Sleep
    elif r == 2 and f in [2, 3]:
        return "About to Sleep"

    # Can't Lose Them
    elif r == 1 and f in [4, 5]:
        return "Can't Lose Them"

    # At Risk
    elif r == 2 and f in [4, 5]:
        return "At Risk"

    # Hibernating
    elif r == 1 and f in [1, 2, 3]:
        return "Hibernating"

    # Low Frequency Recent
    elif r == 3 and f == 1:
        return "About to Sleep"

    # Low Frequency Older
    elif r == 2 and f == 1:
        return "Hibernating"

    else:
        return "Hibernating"


customer_segmentation_monthly["customer_segment"] = (
    customer_segmentation_monthly.apply(
        lambda row: assign_segment(
            row["recency_score"],
            row["frequency_score"]
        ),
        axis=1
    )
)

In [50]:
# def assign_segment(r, f):
#     # Highly Active/Recent Customers
#     if r == 5 and f >= 4: 
#         return "Champions"
#     if r in [4, 5] and f in [2, 3]: 
#         return "Potential Loyalists"
#     if r == 5 and f == 1: 
#         return "New Customers"
#     if r == 4 and f == 1: 
#         return "Promising"
    
#     # Mid-Active/Aging Customers
#     if r == 3 and f >= 3: 
#         return "Loyal Customers"
#     if r == 3 and f in [1, 2]: 
#         return "Need Attention"
#     if r == 2 and f >= 3: 
#         return "At Risk"
#     if r == 2 and f in [1, 2]: 
#         return "About to Sleep"
    
#     # Inactive/Lost Customers
#     if r == 1 and f >= 4: 
#         return "Can't Lose Them"
    
#     # Catch-all for R=1 low frequency or any missing combinations
#     return "Hibernating"

# # Apply the function to your dataframe
# customer_segmentation_monthly["customer_segment"] = (
#     customer_segmentation_monthly.apply(
#         lambda row: assign_segment(
#             row["recency_score"],
#             row["frequency_score"]
#         ),
#         axis=1
#     )
# )
def assign_segment(r, f):
    """
    100% Error-free RFM segmentation using standard operators.
    No bracket lists are used to prevent syntax rendering issues.
    """
    # Highly Active / Recent Customers
    if r == 5 and f >= 4:
        return "Champions"

    elif r == 4 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f >= 2 and f <= 3:
        return "Potential Loyalists"

    elif r == 5 and f == 1:
        return "New Customers"

    elif r == 4 and f == 1:
        return "Promising"

    # Mid-Active Customers
    elif r == 3 and f >= 3:
        return "Loyal Customers"

    elif r == 3 and f >= 1 and f <= 2:
        return "Need Attention"

    elif r == 2 and f >= 3:
        return "At Risk"

    elif r == 2 and f >= 1 and f <= 2:
        return "About to Sleep"

    # Inactive Customers
    elif r == 1 and f >= 4:
        return "Can't Lose Them"

    else:
        # Catch-all for r=1 with low frequency
        return "Hibernating"


# Apply the logic to your dataframe
customer_segmentation_monthly["customer_segment"] = (
    customer_segmentation_monthly.apply(
        lambda row: assign_segment(
            int(row["recency_score"]),
            int(row["frequency_score"])
        ),
        axis=1
    )
)


In [51]:
customer_segmentation_monthly["customer_segment"].value_counts()

customer_segment
Loyal Customers        157761
Potential Loyalists    126108
Hibernating             95280
At Risk                 94734
Champions               63794
Need Attention          63098
About to Sleep          63085
Can't Lose Them         62528
Promising               31595
New Customers           31110
Name: count, dtype: int64

In [58]:
customer_segmentation_monthly

,customer_unique_id,first_purchase_date,first_purchase_month,snapshot_month_start,snapshot_month_end,monthly_orders,monthly_monetary_value,frequency,monetary_value,last_purchase_date,recency_days,recency_score,frequency_score,monetary_score,rfm_score,customer_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-05-01,2018-05-31,1,141.90,1,141.90,2018-05-10,21,5,5,4,554,Champions
1,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-06-01,2018-06-30,0,0.00,1,141.90,2018-05-10,51,5,5,4,554,Champions
2,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-07-01,2018-07-31,0,0.00,1,141.90,2018-05-10,82,5,5,4,554,Champions
3,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-08-01,2018-08-31,0,0.00,1,141.90,2018-05-10,113,4,5,4,454,Loyal Customers
4,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,2018-05-01,2018-05-01,2018-05-31,1,27.19,1,27.19,2018-05-07,24,5,5,1,551,Champions
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789088,ffffd2657e2aad2907e67c3e9daecbeb,2017-05-02,2017-05-01,2018-04-01,2018-04-30,0,0.00,1,71.56,2017-05-02,363,1,1,2,112,Hibernating
789089,ffffd2657e2aad2907e67c3e9daecbeb,2017-05-02,2017-05-01,2018-05-01,2018-05-31,0,0.00,1,71.56,2017-05-02,394,1,1,2,112,Hibernating
789090,ffffd2657e2aad2907e67c3e9daecbeb,2017-05-02,2017-05-01,2018-06-01,2018-06-30,0,0.00,1,71.56,2017-05-02,424,1,1,2,112,Hibernating
789091,ffffd2657e2aad2907e67c3e9daecbeb,2017-05-02,2017-05-01,2018-07-01,2018-07-31,0,0.00,1,71.56,2017-05-02,455,1,1,2,112,Hibernating


In [53]:
customer_segmentation_monthly.info()

<class 'pandas.DataFrame'>
RangeIndex: 789093 entries, 0 to 789092
Data columns (total 16 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   customer_unique_id         789093 non-null  str           
 1   first_purchase_date        789093 non-null  datetime64[us]
 2   first_purchase_month       789093 non-null  datetime64[us]
 3   snapshot_month_start       789093 non-null  datetime64[us]
 4   snapshot_month_end         789093 non-null  datetime64[us]
 5   monthly_orders             789093 non-null  int64         
 6   monthly_monetary_value     789093 non-null  float64       
 7   cumulative_frequency       789093 non-null  int64         
 8   cumulative_monetary_value  789093 non-null  float64       
 9   last_purchase_date         789093 non-null  datetime64[us]
 10  recency_days               789093 non-null  int64         
 11  recency_score              789093 non-null  int64         
 12 

In [54]:
customer_segmentation_monthly.isna().sum()

customer_unique_id           0
first_purchase_date          0
first_purchase_month         0
snapshot_month_start         0
snapshot_month_end           0
monthly_orders               0
monthly_monetary_value       0
cumulative_frequency         0
cumulative_monetary_value    0
last_purchase_date           0
recency_days                 0
recency_score                0
frequency_score              0
monetary_score               0
rfm_score                    0
customer_segment             0
dtype: int64

In [55]:
customer_segmentation_monthly = customer_segmentation_monthly.rename(
    columns={
        "cumulative_frequency": "frequency",
        "cumulative_monetary_value": "monetary_value"
    }
)

In [56]:
customer_segmentation_monthly.to_csv(
    "customer_segmentation_monthly.csv",
    index=False
)

In [57]:
customer_segmentation_monthly.columns

Index(['customer_unique_id', 'first_purchase_date', 'first_purchase_month',
       'snapshot_month_start', 'snapshot_month_end', 'monthly_orders',
       'monthly_monetary_value', 'frequency', 'monetary_value',
       'last_purchase_date', 'recency_days', 'recency_score',
       'frequency_score', 'monetary_score', 'rfm_score', 'customer_segment'],
      dtype='str')

In [59]:
customer_segmentation_monthly['customer_unique_id'] =='0000366f3b9a7992bf8c76cfdf3221e2	'

0         False
1         False
2         False
3         False
4         False
          ...  
789088    False
789089    False
789090    False
789091    False
789092    False
Name: customer_unique_id, Length: 789093, dtype: bool

In [60]:
customer_id = "0000366f3b9a7992bf8c76cfdf3221e2"

customer_segmentation_monthly.loc[
    customer_segmentation_monthly["customer_unique_id"].str.strip() == customer_id,
    :
]

,customer_unique_id,first_purchase_date,first_purchase_month,snapshot_month_start,snapshot_month_end,monthly_orders,monthly_monetary_value,frequency,monetary_value,last_purchase_date,recency_days,recency_score,frequency_score,monetary_score,rfm_score,customer_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-05-01,2018-05-31,1,141.9,1,141.9,2018-05-10,21,5,5,4,554,Champions
1,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-06-01,2018-06-30,0,0.0,1,141.9,2018-05-10,51,5,5,4,554,Champions
2,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-07-01,2018-07-31,0,0.0,1,141.9,2018-05-10,82,5,5,4,554,Champions
3,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,2018-05-01,2018-08-01,2018-08-31,0,0.0,1,141.9,2018-05-10,113,4,5,4,454,Loyal Customers
